# 第10回 大規模言語モデル — 大きくすると、何が変わるのか

**このノートブックは第10回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 大きさだけが違う3〜4個のモデルで loss を測り、スケーリング則を自分の目で見る
2. 同じ質問への答えが、大きさでどう変わるかを見る
3. 手本を見せるだけで成績が変わること(文脈内学習)を確かめる
4. **「創発」が測り方によって見え方を変えることを確かめる**

所要時間:実習 約30分 / 課題 約40分

> **注意**:合計で数GBのモデルをダウンロードします。時間に余裕をもって実行してください。

## 0. 準備

上から順にセルを実行してください(セルを選んで **Shift + Enter**)。

> **注意**:このノートブックは合計で数GBのモデルをダウンロードします。
> 3〜10分ほどかかるセルがあります。実行中は左の丸印が回ります。

In [ ]:
# 準備(このセルを最初に1回だけ実行)
!pip install -q transformers japanize-matplotlib
import warnings; warnings.filterwarnings("ignore")
import torch, numpy as np
import matplotlib.pyplot as plt, japanize_matplotlib
from transformers import AutoTokenizer, AutoModelForCausalLM
plt.rcParams["font.size"] = 12
トークナイザ = AutoTokenizer.from_pretrained("gpt2")   # 4サイズ共通
print("準備できました")

In [ ]:
# 今日は「同じ作りで大きさだけ違う」4つのモデルを比べます。
# GPT-2 は small / medium / large / xl の4種類が公開されています。
# 作り方・学習データは同じで、大きさだけが違う ── 比較にうってつけです。

モデル名 = ["gpt2", "gpt2-medium", "gpt2-large"]
# ↑ gpt2-xl(15億)も加えられますが、6GB ほどダウンロードします。
#    時間に余裕があれば、下の行のコメントを外してください。
# モデル名.append("gpt2-xl")

読み込み済み = {}
def 読み込む(名):
    if 名 not in 読み込み済み:
        print(f"  {名} を読み込み中…(初回はダウンロードします)")
        m = AutoModelForCausalLM.from_pretrained(名); m.eval()
        読み込み済み[名] = m
    return 読み込み済み[名]

print("比べるモデル:", モデル名)
print("※ 合計で数GBダウンロードします。3〜10分ほどかかります。")

## 1. スケーリング則 — 大きくすると、どうなるか

GPT-2 には **small / medium / large / xl** の4種類があります。
**作り方も学習データも同じで、大きさだけが違います。**だから「大きさの効果」だけを取り出せます。

まず、学習に使われていない英文で「次の語をどれだけ当てられるか」を測ります。

In [ ]:
# ===== ① スケーリング則:大きくすると loss はどうなるか =====
# 学習に使われていない英文を用意し、「次の語をどれだけ当てられるか」で測ります。
本文 = """The library opens at nine in the morning and closes at six in the evening.
Students often come here to read quietly or to prepare for their examinations.
On rainy days the reading room becomes crowded, and finding an empty seat is difficult.
The librarian keeps a small notebook in which she records the titles that people ask for.
She says that the same three books are requested almost every week, year after year.
Yesterday a young man asked for a map of the old city, but no such map remained.
He seemed disappointed, and left without borrowing anything at all.
The next morning he returned with a map of his own, drawn carefully by hand.
He gave it to the librarian and asked her to keep it on the shelf for others."""

ids = トークナイザ.encode(本文, return_tensors="pt")
記録 = []
for 名 in モデル名:
    m = 読み込む(名)
    重み数 = sum(p.numel() for p in m.parameters())
    with torch.no_grad():
        loss = float(m(ids, labels=ids).loss)
        予測 = m(ids).logits[0, :-1].argmax(-1)
    正解率 = float((予測 == ids[0, 1:]).float().mean()) * 100
    記録.append((名, 重み数, loss, 正解率))
    print(f"{名:<14}重み {重み数/1e6:7.1f}M   loss {loss:.3f}   次の語の正解率 {正解率:.1f}%")

print("\n重みを増やすと loss が下がります。これがスケーリング則です。")

In [ ]:
# 両対数のグラフにすると、ほぼ直線になります
重み = np.array([r[1] for r in 記録])
loss = np.array([r[2] for r in 記録])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
axes[0].plot(重み/1e6, loss, "o-", ms=9, lw=2, color="#3A46A8")
for 名, w, l, _ in 記録:
    axes[0].annotate(名.replace("gpt2", "GPT-2"), (w/1e6, l),
                     xytext=(6, 6), textcoords="offset points", fontsize=9)
axes[0].set_xlabel("重みの数(百万個)"); axes[0].set_ylabel("loss(小さいほど良い)")
axes[0].set_title("ふつうの目盛り"); axes[0].grid(alpha=.3)

axes[1].loglog(重み, loss, "o-", ms=9, lw=2, color="#E08A2B")
axes[1].set_xlabel("重みの数(対数)"); axes[1].set_ylabel("loss(対数)")
axes[1].set_title("両対数の目盛り ── ほぼ直線"); axes[1].grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()

# 直線のかたむきを求める
かたむき = np.polyfit(np.log(重み), np.log(loss), 1)[0]
print(f"両対数での直線のかたむき: {かたむき:.3f}")
print(f"→ 重みを10倍にすると、loss はおよそ {10**かたむき:.3f} 倍(約 {(1-10**かたむき)*100:.0f}% 減)")
print()
print("【大事なこと】この直線は、どこまでも続くわけではありません。")
print("下には限界があります(言葉には本質的に予測できない部分があるため)。")
print("それでも、この直線が『次にいくら投資すれば、どれだけ良くなるか』を予測させました。")

> **なぜこの直線が重要だったのか**
>
> 「これだけお金と計算をつぎ込めば、これだけ性能が上がる」と**事前に予測できる**ようになったからです。
> 数百億円規模の投資判断が、この直線の上で行われるようになりました。
> いまの大規模言語モデルは、この予測可能性の上に成り立っています。

## 2. 何が「できる」ようになるのか

loss が下がるとは、具体的に何が変わることでしょうか。
同じ質問を、大きさの違うモデルに聞いてみます。

In [ ]:
# ===== ② 何が「できる」ようになるのか =====
質問 = ["The capital of France is",
        "The largest planet in our solar system is",
        "Two plus three equals",
        "The opposite of hot is",
        "Water freezes at a temperature of"]

for 名 in モデル名:
    m = 読み込む(名)
    print(f"【{名}】")
    for q in 質問:
        qi = トークナイザ.encode(q, return_tensors="pt")
        with torch.no_grad():
            p = torch.softmax(m(qi).logits[0, -1], dim=-1)
        上位 = torch.topk(p, 3)
        答 = ", ".join(f"{トークナイザ.decode([i]).strip()}({float(v)*100:.1f}%)"
                       for i, v in zip(上位.indices,上位.values))
        print(f"   {q:<42}→ {答}")
    print()

print("小さいモデルは the / a / about のような『無難な語』を出しがちです。")
print("大きくなると、Paris や five のような『中身のある語』が出るようになります。")

## 3. 文脈内学習 — 手本を見せると変わるか

大きなモデルには、小さなモデルにはない使い方があります。
**プロンプトに手本を書いておくと、それを真似する**という性質です。

重みは1つも変えません。**書き方を変えるだけ**です。

In [ ]:
# ===== ③ 文脈内学習:手本を見せると変わるか =====
# 学習しなおすのではありません。プロンプトに例を書くだけです。
対 = [("hot","cold"),("big","small"),("fast","slow"),("light","dark"),
      ("happy","sad"),("young","old"),("rich","poor"),("hard","soft"),
      ("long","short"),("high","low"),("wet","dry"),("clean","dirty")]

def 手本を作る(個数, 除く語):
    使う = [p for p in 対 if p[0] != 除く語][:個数]
    return "".join(f"{a} -> {b}\n" for a, b in 使う)

結果 = {}
print(f"{'モデル':<14}{'手本0個':>10}{'手本2個':>10}{'手本5個':>10}")
print("-" * 46)
for 名 in モデル名:
    m = 読み込む(名); 行 = []
    for 手本数 in [0, 2, 5]:
        正答 = 0
        for a, b in 対:
            prompt = 手本を作る(手本数, a) + f"{a} ->"
            pid = トークナイザ.encode(prompt, return_tensors="pt")
            with torch.no_grad():
                p = torch.softmax(m(pid).logits[0, -1], dim=-1)
            出た = トークナイザ.decode([int(p.argmax())]).strip().lower()
            正答 += int(出た == b)
        行.append(正答 / len(対) * 100)
    結果[名] = 行
    print(f"{名:<14}{行[0]:>9.0f}%{行[1]:>9.0f}%{行[2]:>9.0f}%")

print("\n手本を見せるだけで成績が変わるなら、それが『文脈内学習』です。")
print("重みは1つも変わっていません。プロンプトに書いただけです。")

In [ ]:
# グラフで見る
plt.figure(figsize=(7.5, 4.2))
色 = ["#A8AECC", "#5B6BD6", "#3A46A8", "#E08A2B"]
for (名, 値), c in zip(結果.items(), 色):
    plt.plot([0, 2, 5], 値, "o-", lw=2, ms=8, label=名, color=c)
plt.xlabel("プロンプトに書いた手本の数"); plt.ylabel("正解率(%)")
plt.title("手本を見せるだけで、成績は変わるか")
plt.xticks([0, 2, 5]); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("これが第11回『プロンプト』の土台です。")
print("モデルを作りなおさなくても、書き方を変えるだけで結果が変わる ── だから")
print("『プロンプトの書き方』という技能が生まれました。")

## 4. 「創発」は本物か — 測り方を疑う

「ある大きさを超えると、突然できるようになる」── これを **創発(emergence)** と呼びます。
Transformer の解説と同じくらい、よく引用される主張です。

**本当でしょうか。**同じ実験を、2通りの測り方でやってみます。

In [ ]:
# ===== ④ 「創発」は本物か ── 指標の見せかたを疑う =====
# 足し算(答えが1トークンになるものだけ)を解かせます。
# ここでは2通りの測り方をします。
#   ① 完全正解率 …「いちばん確率の高い語が正解か」(0点か1点か)
#   ② 正解に与えた確率の平均 …「正解にどれだけ近づいているか」(なめらか)

問題 = [(f"Question: What is {a} plus {b}? Answer: {a} plus {b} is", str(a+b))
        for a in range(1, 10) for b in range(1, 10) if a + b < 10]
print(f"問題数: {len(問題)}\n")

創発記録 = []
print(f"{'モデル':<14}{'完全正解率':>12}{'正解に与えた確率':>18}{'正解の平均順位':>16}")
print("-" * 62)
for 名 in モデル名:
    m = 読み込む(名)
    正答 = 0; 確率合計 = 0.0; 順位合計 = 0
    for q, 答 in 問題:
        pid = トークナイザ.encode(q, return_tensors="pt")
        with torch.no_grad():
            p = torch.softmax(m(pid).logits[0, -1], dim=-1)
        正解id = トークナイザ.encode(" " + 答)[0]
        正答 += int(int(p.argmax()) == 正解id)
        確率合計 += float(p[正解id])
        順位合計 += int((p > p[正解id]).sum()) + 1
    n = len(問題)
    重み数 = sum(x.numel() for x in m.parameters())
    創発記録.append((名, 重み数, 正答/n*100, 確率合計/n*100, 順位合計/n))
    print(f"{名:<14}{正答/n*100:>11.1f}%{確率合計/n*100:>17.2f}%{順位合計/n:>15.1f}位")

In [ ]:
# 2つの測り方を並べる
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
w = [r[1]/1e6 for r in 創発記録]

axes[0].plot(w, [r[2] for r in 創発記録], "o-", ms=9, lw=2.5, color="#E08A2B")
axes[0].set_xlabel("重みの数(百万個)"); axes[0].set_ylabel("完全正解率(%)")
axes[0].set_title("① 完全正解率(0点か1点か)"); axes[0].grid(alpha=.3)

axes[1].plot(w, [r[3] for r in 創発記録], "o-", ms=9, lw=2.5, color="#3A46A8")
axes[1].set_xlabel("重みの数(百万個)"); axes[1].set_ylabel("正解に与えた確率(%)")
axes[1].set_title("② 正解に与えた確率(なめらか)"); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("同じモデル、同じ問題、同じ実験です。違うのは『測り方』だけ。")
print()
print("① は段差に見えやすく、② はなめらかに見えやすい。")
print("『ある大きさを超えると突然できるようになる(創発)』という主張の一部は、")
print("厳しい測り方を使ったことによる見かけ上の段差だ ── という指摘があります。")
print("(Schaeffer ら 2023「Are Emergent Abilities of Large Language Models a Mirage?」)")
print()
print("どちらの測り方が正しいのでしょうか。実は、目的によります。")
print("『答えとして使えるか』を知りたいなら ①。『内部で何が育っているか』を見たいなら ②。")

> **今日いちばん持ち帰ってほしいこと**
>
> 同じモデル、同じ問題、同じ実験でも、**測り方を変えると結論が変わって見えます。**
>
> 「段差がある」と言う人と「なめらかだ」と言う人が、同じデータを見ていることがあります。
> グラフを見たら「**何を、どう測ったグラフか**」を確かめてください。

## 5. 自分で試す

**壊しても大丈夫**です(開き直せば元に戻ります)。

In [ ]:
# ===== 自分で試す(課題用)=====
# (a) 自分の質問で、大きさによる違いを見る
私の質問 = [
    "The president of the United States lives in",
    "The chemical symbol for gold is",
    "Nine minus four equals",
]
for 名 in モデル名:
    m = 読み込む(名)
    print(f"【{名}】")
    for q in 私の質問:
        qi = トークナイザ.encode(q, return_tensors="pt")
        with torch.no_grad():
            p = torch.softmax(m(qi).logits[0, -1], dim=-1)
        上位 = torch.topk(p, 3)
        print(f"   {q:<44}→ " + ", ".join(
            f"{トークナイザ.decode([i]).strip()}({float(v)*100:.1f}%)"
            for i, v in zip(上位.indices, 上位.values)))
    print()

# (b) 自分で手本を作って、文脈内学習を試す
私の手本 = "apple -> fruit\ncarrot -> vegetable\nbanana -> fruit\nspinach ->"
for 名 in モデル名:
    m = 読み込む(名)
    pid = トークナイザ.encode(私の手本, return_tensors="pt")
    with torch.no_grad():
        p = torch.softmax(m(pid).logits[0, -1], dim=-1)
    print(f"{名:<14}→ {トークナイザ.decode([int(p.argmax())]).strip()}"
          f" ({float(p.max())*100:.1f}%)")

## 6. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:手で計算する(電卓可)
あるモデルの loss を測ったところ、次のようになりました。

| 重みの数 | loss |
|---|---|
| 1億個 | 3.00 |
| 10億個 | 2.70 |
| 100億個 | 2.43 |

- (a) 重みを10倍にするたび、loss は何倍になっていますか
- (b) この調子が続くとすると、1兆個(1万億)のとき loss はいくつになりますか
- (c) その予測を、そのまま信じてよいですか。**信じてはいけない理由**を書きなさい

### 課題2:ノートブックを動かして報告する
- (a) 重みの数と loss のグラフ(ふつうの目盛りと両対数の2枚)
- (b) 自分で用意した質問3つに対する、各サイズの答えの表
- (c) 「完全正解率」と「正解に与えた確率」の2枚のグラフを貼り、**印象の違い**を説明する

### 課題3:次の主張を批判する(300字以上)

> 「スケーリング則によれば、モデルを大きくすれば性能は上がり続けます。
> したがって、計算資源に投資し続ければ、いずれ人間を超える知能に到達します。
> これは実験で確かめられた法則であり、議論の余地はありません。」

考える材料:
- 「法則」という言葉は適切ですか。これは何を観察した結果ですか
- 直線はどこまでも続きますか。下限はありませんか
- 「loss が下がる」ことと「人間を超える」ことは、同じ意味ですか
- 大きさ以外に必要なものはありませんか(データ、電力、コスト)

**加点**:「では、どこまでなら言えるか」まで書けると評価が上がります。

---

採点は、正解かどうかよりも **試したことと考えたこと** を見ます。
課題2(c) は、どちらのグラフが「正しい」かではなく、**なぜ印象が違うのか**を説明できているかを見ます。